In [21]:
# !pip install requests um Package zu installieren
# !pip install beautifulsoup4 um Package zu installieren
# !pip install pandas um Package zu installieren

import requests
import bs4
from bs4 import BeautifulSoup
import pandas

# print(requests.__version__) um Version zu bekommen, zeigt ob Package wirklich installiert wurde
# print(bs4.__version__) um Version zu bekommen, zeigt ob Package wirklich installiert wurde
# print(pandas.__version__) um Version zu bekommen, zeigt ob Package wirklich installiert wurde

Teams_dict = { # Dictionary für alle Teams
    "Team Name": [],
    "Year": [],
    "Wins": [],
    "Losses": [],
    "OT Losses": [],
    "Win %": [],
    "Goals For (GF)": [],
    "Goals Against (GA)": [],
    "+/-": []
}

for Seite in range(1, 25): # Anzahl der Seiten in der Tabelle
    if Seite == 1:
        url = "https://www.scrapethissite.com/pages/forms/"
    else:
        url = f"https://www.scrapethissite.com/pages/forms/?page_num={Seite}" # Findet alle weiteren Seiten der Tabelle
    
    response = requests.get(url)
    if response.status_code != 200: # Wenn != 200, dann kann Seite nicht korrekt geladen werden
        print(f"Seite {Seite} konnte nicht geladen werden!")

    soup = BeautifulSoup(response.text, 'html.parser')
    table = soup.find('table', class_='table')  # Findet die Tabelle, aus denen die Daten gescrapted werden sollen
    Zeilen = table.find_all('tr')  # Findet alle Zeilen 

    for Zeile in Zeilen[1:]:  # Überspringe die erste Zeile, die nur den Namen der Spalte hat
        Zelle = Zeile.find_all('td')  
        if Zelle:  # nur Zeilen die Daten enthalten
            Teams_dict["Team Name"].append(Zelle[0].text.strip())  
            Teams_dict["Year"].append(Zelle[1].text.strip())
            Teams_dict["Wins"].append(Zelle[2].text.strip())
            Teams_dict["Losses"].append(Zelle[3].text.strip())
            Teams_dict["OT Losses"].append(Zelle[4].text.strip())
            Teams_dict["Win %"].append(Zelle[5].text.strip())
            Teams_dict["Goals For (GF)"].append(Zelle[6].text.strip())
            Teams_dict["Goals Against (GA)"].append(Zelle[7].text.strip())
            Teams_dict["+/-"].append(Zelle[8].text.strip())

# Alle Daten in DataFrame
df = pandas.DataFrame(Teams_dict)
df.to_csv("data.csv", index=False) # Index = False, damit die Nummerierung nicht mit in die CSV Datei geht

df = pandas.read_csv("data.csv")


print("Frage: Who made the most \"wins\" in 1990, 2000, and 2010?")

Jahresanzahl_Frage_1 = [1990, 2000, 2010] # Liste mit den Jahreszahlen, die von Interesse sind

for Jahr in Jahresanzahl_Frage_1: # Schleife, um durch alle Jahre in der Variable Jahresanzahl durchzugehen
    Jahre = df[df['Year'] == Jahr] # Filtern auf Jahr das von Interesse ist
    Meisten_siege = Jahre['Wins'].idxmax() # Findet größten Wert, also meiste Siege in diesem Jahr
    Meisten_siege_teams = df.loc[Meisten_siege] # Nimmt die ganze Zeile, die bei Wins den Wert aus Meisten_siege hat
    print(f"Das Team mit den meisten Siegen im Jahre {Jahr} ist {Meisten_siege_teams['Team Name']} mit {Meisten_siege_teams['Wins']} Siegen.")

print() # Abstand zwischen den Fragen erstellen

print("Frage: How many teams participated in 1991, 2001, and 2011?")

Jahresanzahl_Frage_2 = [1991, 2001, 2011] # Liste mit den Jahreszahlen, die von Interesse sind

for Jahr in Jahresanzahl_Frage_2: # Schleife, um durch alle Jahre in der Variable Jahresanzahl durchzugehen
    Jahre = df[df['Year'] == Jahr]
    Anzahl_Teams = Jahre.shape[0]   # Zählt die Zeilen im Jahr das grad in der Schleife dran ist und damit die Anazhl der Teams in dem Jahr
    print(f"Im Jahre {Jahr} haben {Anzahl_Teams} Teams teilgenommen.")

Frage: Who made the most "wins" in 1990, 2000, and 2010?
Das Team mit den meisten Siegen im Jahre 1990 ist Chicago Blackhawks mit 49 Siegen.
Das Team mit den meisten Siegen im Jahre 2000 ist Colorado Avalanche mit 52 Siegen.
Das Team mit den meisten Siegen im Jahre 2010 ist Vancouver Canucks mit 54 Siegen.

Frage: How many teams participated in 1991, 2001, and 2011?
Im Jahre 1991 haben 22 Teams teilgenommen.
Im Jahre 2001 haben 30 Teams teilgenommen.
Im Jahre 2011 haben 30 Teams teilgenommen.
